<a href="https://colab.research.google.com/github/navsingh2024-bit/Gen_AI-Property_price_prediction-/blob/main/Gen_AI_House_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏡 Project 9: Intelligent Property Price Prediction
**Milestone 1: Classical Machine Learning on Gurgaon Real Estate**

**Objective:**
Develop an end-to-end machine learning pipeline to predict real estate prices based on proprietary property attributes (BHK, Luxury Category, Sector Location).

**Technical Stack Compliance:**
* **Dataset:** Custom Gurgaon Properties (Post-Feature Selection)
* **Algorithm:** Random Forest Regressor
* **Workflow:** Scikit-Learn Pipelines (StandardScaler + OneHotEncoder)
* **Evaluation:** R-Squared ($R^2$), Mean Absolute Error (MAE)


In [ ]:
# Phase 0: Environment Stabilization
# CRITICAL: This prevents version mismatch errors during Streamlit deployment.
!pip install scikit-learn==1.3.2 -U
!pip install pandas numpy matplotlib seaborn joblib

In [ ]:
# Phase 1: Environment Setup & Data Ingestion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn Modules
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics

# Load the proprietary dataset
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv')

print("--- Data Ingestion Complete ---")
print(f"Total Properties: {df.shape[0]}")
print(f"Total Features: {df.shape[1]}")

### Phase 2: Exploratory Data Analysis (EDA)
Establishing the statistical baseline required for the Mid-Sem Project Report. Note: Categorical features are excluded from the correlation heatmap to maintain matrix integrity.

In [ ]:
# 1. Statistical Summary
print("--- Summary Statistics ---")
display(df.describe())

# 2. Identify Numeric Columns for Correlation
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# 3. Feature Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='Blues', fmt=".2f", annot_kws={"size": 10})
plt.title("Numeric Feature Correlation Matrix (Gurgaon Properties)")
plt.show()

### Phase 3: Model Architecture & Scikit-Learn Pipeline
To handle a mix of categorical (e.g., Sector, Property Type) and numerical data without data leakage, we utilize a `ColumnTransformer` inside the `Pipeline`.
1. **Numeric Features**: Scaled using `StandardScaler`.
2. **Categorical Features**: Encoded using `OneHotEncoder` (configured to ignore unknown sectors in production).
3. **Model**: `RandomForestRegressor` (Hyperparameters constrained to ensure deployment file size remains < 100MB).

In [ ]:
# 1. Separate Features (X) and Target (y)
X = df.drop(columns=['price'])
y = df['price']

# 2. Dynamically identify categorical vs numerical columns
numeric_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# 3. Build the Preprocessing Engine
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# 4. Construct the Final Pipeline (Fulfills Rubric Requirement)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('rf_model', RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1 # Uses all cores for fast training
    ))
])

# 5. Train/Test Split & Execution
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Initiating Pipeline Training on Gurgaon Dataset...")
pipeline.fit(X_train, y_train)
print("Model Training Complete.")

### Phase 4: Model Evaluation
Testing the predictive accuracy on the unseen 20% validation set using standard regression metrics.

In [ ]:
# Generate predictions on unseen test data
y_pred = pipeline.predict(X_test)

# Calculate Core Metrics
r2 = metrics.r2_score(y_test, y_pred)
mae = metrics.mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(metrics.mean_squared_error(y_test, y_pred))

print('--- Model Performance on Test Set (Copy to Report) ---')
print(f'R-Squared (Variance Explained): {r2:.4f}')
print(f'Mean Absolute Error (MAE):      {mae:.4f} Crores')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f} Crores')

# Visualizing Actual vs. Predicted Prices
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='teal')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Property Price (Crores)")
plt.ylabel("Predicted Property Price (Crores)")
plt.title("Actual vs. Predicted Prices (Gurgaon Properties)")
plt.show()

### Phase 5: Serialization & Deployment Prep
Serializing the trained pipeline utilizing compression. This ensures the `.pkl` artifact complies with GitHub's strict 100MB repository limits for Streamlit deployment.

In [ ]:
# Save the model for UI integration
model_filename = 'rf_gurgaon_pipeline_final.pkl'
joblib.dump(pipeline, model_filename, compress=3)

print(f"Pipeline successfully serialized and saved as '{model_filename}'")
print("Ready for Streamlit UI integration.")